In [36]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import glob
import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, train_test_split

## Functions

In [49]:
def train_svm_k_folds(df, n_folds):
    X = df.drop("label", axis=1)
    y= df["label"]

    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=100)
    kf.get_n_splits(X)

    pipeline_svm = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(random_state=42))
    ])

    param_grid_svm = {
        'svm__C': [0.001, 0.1, 1, 10, 100],        
        'svm__gamma': ['auto'],
        'svm__kernel': ['rbf', 'linear']         
    }

    best_svm_params, best_svm_score = fit_grid_search(pipeline_svm, param_grid_svm, kf, X, y)
    return best_svm_params, best_svm_score
    


In [50]:
def train_knn_k_folds(df, n_folds):
    X = df.drop("label", axis=1)
    y= df["label"]

    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=100)
    kf.get_n_splits(X)

    pipeline_knn = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier())
    ])

    param_grid_knn = {

        "knn__n_neighbors":[1, 3, 5, 9, 13, 17],
        "knn__metric":['euclidean', 'manhattan']         
    }

    best_knn_params, best_knn_score = fit_grid_search(pipeline_knn, param_grid_knn, kf, X, y)
    return best_knn_params, best_knn_score
    
    

In [51]:
def fit_grid_search(pipeline, param_grid, kf, X, y):
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=kf,
        scoring='accuracy',
        verbose=0,         
        n_jobs=-1 
    )
    grid_search.fit(X, y)

    return grid_search.best_params_, grid_search.best_score_

## Code

In [52]:
features_dirs = [Path("lbp_features"), Path("glcm_features")]

In [53]:
results_dict = {}

In [54]:
for feature_dir in features_dirs:
    print(f"Working on folder: {Path(feature_dir)}")
    for file in tqdm.tqdm(glob.glob(os.path.join(feature_dir, "*.csv"))):
        if Path(feature_dir).name == "lbp_features":
            df = pd.read_csv(file, index_col=0)
            df.drop("filename", axis=1, inplace=True)
            
        else:
            df = pd.read_csv(file)

        if Path(file).name not in results_dict:
            results_dict[Path(file).name] = {}
        
        params, score = train_svm_k_folds(df, 10)
        results_dict[Path(file).name]['svm'] = {'params': params, 'score': score}

        params, score = train_knn_k_folds(df, 10)
        results_dict[Path(file).name]['knn'] = {'params': params, 'score': score}

Working on folder: lbp_features


100%|██████████| 10/10 [11:26<00:00, 68.67s/it]


Working on folder: glcm_features


100%|██████████| 7/7 [04:46<00:00, 40.96s/it]


In [56]:
records = []
for filename, models in results_dict.items():
    for model_name, results in models.items():
        records.append({
            'feature_set': filename,
            'model': model_name,
            'best_score': results['score'],
            'best_params': results['params']
        })

# Create the final summary DataFrame
summary_df = pd.DataFrame(records)

print("\n--- Final Summary DataFrame ---")
summary_df


--- Final Summary DataFrame ---


,feature_set,model,best_score,best_params
0,lbp_16_2_nri_uniform_4x4.csv,svm,0.430,"{'svm__C': 10, 'svm__gamma': 'auto', 'svm__ker..."
1,lbp_16_2_nri_uniform_4x4.csv,knn,0.322,"{'knn__metric': 'manhattan', 'knn__n_neighbors..."
2,lbp_16_2_uniform_3x3.csv,svm,0.326,"{'svm__C': 0.1, 'svm__gamma': 'auto', 'svm__ke..."
3,lbp_16_2_uniform_3x3.csv,knn,0.262,"{'knn__metric': 'manhattan', 'knn__n_neighbors..."
4,lbp_16_3_uniform_3x3.csv,svm,0.343,"{'svm__C': 0.1, 'svm__gamma': 'auto', 'svm__ke..."
5,lbp_16_3_uniform_3x3.csv,knn,0.260,"{'knn__metric': 'euclidean', 'knn__n_neighbors..."
6,lbp_8_1_nri_uniform_2x2.csv,svm,0.414,"{'svm__C': 100, 'svm__gamma': 'auto', 'svm__ke..."
7,lbp_8_1_nri_uniform_2x2.csv,knn,0.345,"{'knn__metric': 'manhattan', 'knn__n_neighbors..."
8,lbp_8_1_nri_uniform_4x4.csv,svm,0.422,"{'svm__C': 10, 'svm__gamma': 'auto', 'svm__ker..."
9,lbp_8_1_nri_uniform_4x4.csv,knn,0.334,"{'knn__metric': 'manhattan', 'knn__n_neighbors..."


In [57]:
summary_df.to_csv("./sumario_resultados.csv")

In [60]:
df = pd.read_csv("./lbp_features/lbp_16_2_uniform_3x3.csv",index_col=0)

In [66]:
X = df.drop(["label", "filename"], axis=1)
y = df["label"]

le = LabelEncoder()
y = le.fit_transform(y)

In [67]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

svm = SVC()

param_grid = {
        'C': [0.001, 0.1, 1, 10, 100],        
        'gamma': ['auto'],
        'kernel': ['rbf', 'linear']         
    }

grid_search = GridSearchCV(
        estimator=svm,
        param_grid=param_grid,
        cv=3,
        scoring='accuracy',
        verbose=0,         
        n_jobs=-1 
    )

grid_search.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.001, 0.1, 1, 10, 100], 'gamma': ['auto'],
                         'kernel': ['rbf', 'linear']},
             scoring='accuracy')

In [69]:
best_svm = grid_search.best_estimator_

In [70]:
y_pred = best_svm.predict(X_test)



'              precision    recall  f1-score   support\n\n           0       0.14      0.09      0.11        35\n           1       0.30      0.70      0.42        20\n           2       0.30      0.22      0.25        37\n           3       0.35      0.26      0.30        34\n           4       0.20      0.17      0.18        24\n           5       0.31      0.12      0.17        33\n           6       0.52      0.37      0.43        30\n           7       0.19      0.65      0.29        23\n           8       0.29      0.17      0.22        29\n           9       0.44      0.34      0.39        35\n\n    accuracy                           0.28       300\n   macro avg       0.30      0.31      0.28       300\nweighted avg       0.31      0.28      0.27       300\n'

In [72]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.14      0.09      0.11        35
           1       0.30      0.70      0.42        20
           2       0.30      0.22      0.25        37
           3       0.35      0.26      0.30        34
           4       0.20      0.17      0.18        24
           5       0.31      0.12      0.17        33
           6       0.52      0.37      0.43        30
           7       0.19      0.65      0.29        23
           8       0.29      0.17      0.22        29
           9       0.44      0.34      0.39        35

    accuracy                           0.28       300
   macro avg       0.30      0.31      0.28       300
weighted avg       0.31      0.28      0.27       300

